# NIH ChestX-ray14 Group A: DenseNet-121

## tl;dr

This notebook computes only `condition_1_original` (Group A): raw DenseNet-121 scores, the original held-out age distribution, no calibration, and no age-standardization weights. Thresholds are selected per finding from each calibration split using frozen Youden's J and then applied unchanged to that seed's held-out test patients.

The notebook never substitutes ResNet predictions, chooses thresholds from test outcomes, or fabricates results when inputs are missing.

## Context & Methods

### Key assumptions

- FNR is `FN / (TP + FN)` and is computed separately for recorded female (`F`) and male (`M`) patients.
- The signed gap is `S = FNR(female) - FNR(male)`.
- `NIH_14_pooled` is micro-pooled across all 14 finding/image pairs: total false negatives divided by total positive cases.
- Bootstrap resampling is clustered by `Patient ID`; all rows for a sampled patient remain together.
- The strict result table keeps the shared eight-column schema. CI columns are written separately.

### Kaggle input contract

Upload one Dataset containing this notebook, `group_a_densenet.py`, the shared modules, metadata, frozen split files, and the 12 DenseNet prediction shards. The NIH image Dataset remains separate and is not needed when using these precomputed scores.

In [ ]:
%pip install -q numpy pandas pyarrow

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Edit only this slug when the Kaggle control Dataset has a different name.
KAGGLE_CONTROL_ROOT = Path('/kaggle/input/datasets/rihsav/nih-chestxray14-group-a-densenet-inputs/kaggle_group_a_densenet_input')
CONTROL_ROOT = KAGGLE_CONTROL_ROOT if KAGGLE_CONTROL_ROOT.exists() else Path('.')
OUTPUT_ROOT = (Path('/kaggle/working/group_a_densenet')
               if Path('/kaggle/working').exists()
               else Path('outputs/group_a_densenet'))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CONTROL_ROOT.resolve()))

METADATA_PATH = CONTROL_ROOT / 'metadata_clean.csv'
PREDICTION_GLOB = CONTROL_ROOT / 'predictions' / 'scores_densenet_all_tar*.parquet'
SPLITS_ROOT = CONTROL_ROOT / 'splits'
SEEDS_PATH = SPLITS_ROOT / 'seeds.txt'
FROZEN_SEEDS = [
    3658676649, 768519171, 113462462, 2748406118, 1569714665,
    2006902500, 342858866, 1591287646, 2763601433, 1524358342,
]
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED_BASE = 20260823
print('CONTROL_ROOT:', CONTROL_ROOT.resolve())
print('OUTPUT_ROOT:', OUTPUT_ROOT.resolve())

In [ ]:
import analysis_data
import group_a_densenet as group_a
import results_schema
import results_writer
print('Loaded control modules from:', CONTROL_ROOT)
print('Loaded DenseNet helper:', group_a.__file__)

EXPECTED_FINDINGS = list(group_a.NIH_FINDING_NAMES)
EXPECTED_ALL_LABELS = list(group_a.ALL_FINDING_NAMES)

def _first_column(frame, candidates, required=True):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    if required:
        raise KeyError(f'Missing one of required columns: {candidates}')
    return None

def load_and_normalize_metadata(path):
    frame = pd.read_csv(path, dtype=str)
    image_col = _first_column(frame, ['Image Index', 'image_index'])
    patient_col = _first_column(frame, ['Patient ID', 'patient_id'])
    sex_col = _first_column(frame, ['sex', 'Patient Sex', 'Patient Gender', 'Gender'])
    labels_col = _first_column(frame, ['Finding Labels', 'finding_labels'])
    frame = frame.rename(columns={
        image_col: 'Image Index',
        patient_col: 'Patient ID',
        sex_col: 'sex',
        labels_col: 'Finding Labels',
    })
    frame['Image Index'] = frame['Image Index'].astype(str).str.strip()
    frame['Patient ID'] = frame['Patient ID'].astype(str).str.strip()
    frame['sex'] = frame['sex'].astype(str).str.strip().str.upper().str[0]
    if frame['Image Index'].duplicated().any():
        raise ValueError('Metadata contains duplicate Image Index values.')
    if not set(frame['sex'].dropna()).issubset({'F', 'M'}):
        raise ValueError('Metadata sex values must be exactly F or M after normalization.')
    # Re-parse exact pipe-delimited tokens; do not trust precomputed column order.
    def has_finding(text, finding):
        tokens = {token.strip() for token in str(text).split('|')}
        return int(finding in tokens)
    for finding in EXPECTED_FINDINGS:
        frame[f'{finding}_label'] = frame['Finding Labels'].map(
            lambda text, finding=finding: has_finding(text, finding)
        )
    return frame

def read_parquet_metadata(path):
    raw = pq.read_metadata(path).metadata or {}
    metadata = {}
    for key, value in raw.items():
        key = key.decode('utf-8') if isinstance(key, bytes) else str(key)
        value = value.decode('utf-8') if isinstance(value, bytes) else value
        try:
            metadata[key] = json.loads(value)
        except (TypeError, json.JSONDecodeError):
            metadata[key] = value
    return metadata

EXPECTED_PREDICTION_NAMES = [f'scores_densenet_all_tar{index:02d}.parquet' for index in range(12)]
PREDICTION_PATHS = [CONTROL_ROOT / 'predictions' / name for name in EXPECTED_PREDICTION_NAMES]

def required_input_report():
    missing = []
    if not METADATA_PATH.is_file():
        missing.append(str(METADATA_PATH))
    for prediction_path in PREDICTION_PATHS:
        if not prediction_path.is_file():
            missing.append(str(prediction_path))
    if not SEEDS_PATH.is_file():
        missing.append(str(SEEDS_PATH))
    for seed in FROZEN_SEEDS:
        for name in ('calibration_patients.csv', 'test_patients.csv'):
            path = SPLITS_ROOT / f'seed_{seed}' / name
            if not path.is_file():
                missing.append(str(path))
    return missing

## Data

### 1. Validate inputs

This cell intentionally stops before loading or calculating anything when a required artifact is absent.

In [ ]:
missing_inputs = required_input_report()
validation = {
    'condition': results_schema.CONDITION_1_ORIGINAL,
    'required_findings': EXPECTED_FINDINGS,
    'frozen_seeds': FROZEN_SEEDS,
    'metadata_path': str(METADATA_PATH),
    'prediction_paths': [str(path) for path in PREDICTION_PATHS],
    'splits_root': str(SPLITS_ROOT),
    'missing_inputs': missing_inputs,
    'ready_for_computation': not missing_inputs,
}
(OUTPUT_ROOT / 'input_validation.json').write_text(
    json.dumps(validation, indent=2), encoding='utf-8'
)
print(json.dumps(validation, indent=2))
RUN_ANALYSIS = not missing_inputs
if not RUN_ANALYSIS:
    print('BLOCKED: computation skipped; upload the listed inputs and rerun.')

In [ ]:
if RUN_ANALYSIS:
    metadata = load_and_normalize_metadata(METADATA_PATH)
    prediction_metadata = read_parquet_metadata(PREDICTION_PATHS[0])
    provenance_keys = (
        'weights', 'backbone', 'raw_model_labels', 'output_labels',
        'label_order_validated', 'input_size', 'preprocessing', 'score_definition',
    )
    shard_frames = []
    for prediction_path in PREDICTION_PATHS:
        shard_metadata = read_parquet_metadata(prediction_path)
        for key in provenance_keys:
            if shard_metadata.get(key) != prediction_metadata.get(key):
                raise ValueError(f'Prediction provenance differs across shards for {key}: {prediction_path}')
        shard = pd.read_parquet(prediction_path)
        shard_frames.append(group_a.validate_densenet_prediction_frame(shard, metadata=shard_metadata))
    predictions = pd.concat(shard_frames, ignore_index=True)
    if predictions['Image Index'].duplicated().any():
        raise ValueError('DenseNet prediction shards contain duplicate Image Index values.')
    metadata_ids = set(metadata['Image Index'])
    prediction_ids = set(predictions['Image Index'])
    excluded_prediction_ids = sorted(prediction_ids - metadata_ids)
    if excluded_prediction_ids:
        print('Excluding prediction rows absent from metadata_clean.csv:', len(excluded_prediction_ids))
        print('These are expected rows dropped by the documented age-cleaning rule.')
    predictions = predictions[predictions['Image Index'].isin(metadata_ids)].copy()
    prediction_columns = ['Image Index', *EXPECTED_ALL_LABELS]
    score_columns = predictions[prediction_columns].rename(
        columns={finding: f'{finding}_score' for finding in EXPECTED_ALL_LABELS}
    )
    assembled = analysis_data.assemble_metadata_predictions(
        metadata, score_columns
    )
    if len(assembled) != len(metadata):
        raise ValueError('DenseNet predictions do not cover every cleaned metadata row.')
    print('metadata rows:', len(metadata))
    print('prediction shard rows:', sum(len(frame) for frame in shard_frames))
    print('excluded prediction rows:', len(excluded_prediction_ids))
    print('joined cleaned rows:', len(assembled))
    print('validated model:', prediction_metadata['weights'], prediction_metadata['backbone'])
else:
    metadata = predictions = assembled = None
    excluded_prediction_ids = []

In [ ]:
if RUN_ANALYSIS:
    split_sets = {}
    for seed in FROZEN_SEEDS:
        split_indices = analysis_data.load_split_indices(
            SPLITS_ROOT / f'seed_{seed}' / 'calibration_patients.csv',
            SPLITS_ROOT / f'seed_{seed}' / 'test_patients.csv',
            patient_id_column='Patient ID',
        )
        calibration_ids = set(split_indices.calibration['Patient ID'].astype(str))
        test_ids = set(split_indices.test['Patient ID'].astype(str))
        split_sets[seed] = {'calibration': calibration_ids, 'test': test_ids}
    print('validated disjoint calibration/test patient IDs for', len(split_sets), 'seeds')
else:
    split_sets = {}

## Results

### 2. Compute Group A across all frozen seeds

Only the raw-score condition is evaluated here. Each seed gets its own calibration-derived threshold vector and its own patient-clustered bootstrap.

In [ ]:
if RUN_ANALYSIS:
    all_strict_rows = []
    all_ci_rows = []
    thresholds_by_seed = {}
    for seed in FROZEN_SEEDS:
        print(f'Starting seed {seed}: bootstrap {N_BOOTSTRAP} resamples', flush=True)
        calibration = assembled[assembled['Patient ID'].isin(split_sets[seed]['calibration'])].copy()
        test = assembled[assembled['Patient ID'].isin(split_sets[seed]['test'])].copy()
        if calibration.empty or test.empty:
            raise ValueError(f'Seed {seed} has an empty calibration or test table.')
        thresholds = group_a.choose_frozen_thresholds(
            calibration,
            EXPECTED_FINDINGS,
            label_column=lambda finding: f'{finding}_label',
            score_column=lambda finding: f'{finding}_score',
        )
        thresholds_by_seed[str(seed)] = thresholds
        ci_rows = group_a.compute_group_a_bootstrap_rows(
            test,
            finding_names=EXPECTED_FINDINGS,
            thresholds=thresholds,
            label_column=lambda finding: f'{finding}_label',
            score_column=lambda finding: f'{finding}_score',
            patient_id_column='Patient ID',
            sex_column='sex',
            female_value='F',
            male_value='M',
            n_resamples=N_BOOTSTRAP,
            confidence_level=0.95,
            random_seed=BOOTSTRAP_SEED_BASE + int(seed),
        )
        for row in ci_rows:
            row['split_seed'] = int(seed)
        all_ci_rows.extend(ci_rows)
        all_strict_rows.extend(
            [{column: row[column] for column in results_schema.REQUIRED_COLUMNS} for row in ci_rows]
        )
        print(f'Finished seed {seed}', flush=True)

    strict_rows = results_schema.validate_records(all_strict_rows)
    ci_rows = results_writer.validate_result_rows(all_ci_rows)
    strict_path = OUTPUT_ROOT / 'group_a_densenet_condition_1_original_strict.csv'
    ci_path = OUTPUT_ROOT / 'group_a_densenet_condition_1_original_with_ci.csv'
    results_writer.write_results_csv(strict_path, strict_rows)
    results_writer.write_results_csv(ci_path, ci_rows)
    (OUTPUT_ROOT / 'thresholds_by_seed.json').write_text(
        json.dumps(thresholds_by_seed, indent=2, sort_keys=True), encoding='utf-8'
    )
    manifest = {
        'dataset': 'NIH ChestX-ray14',
        'backbone': group_a.DENSENET121_WEIGHTS,
        'condition': results_schema.CONDITION_1_ORIGINAL,
        'threshold_rule': "Youden's J; calibration split only; highest threshold on ties",
        'pooled_fnr': 'micro-FNR across all 14 finding/image pairs',
        'bootstrap': {'cluster_column': 'Patient ID', 'draws': N_BOOTSTRAP, 'confidence_level': 0.95, 'interval': 'percentile'},
        'inputs': {'metadata': str(METADATA_PATH), 'prediction_shards': [str(path) for path in PREDICTION_PATHS], 'splits_root': str(SPLITS_ROOT)},
        'excluded_prediction_rows_absent_from_clean_metadata': excluded_prediction_ids,
        'outputs': {'strict': str(strict_path), 'ci_enriched': str(ci_path)},
    }
    (OUTPUT_ROOT / 'run_manifest.json').write_text(
        json.dumps(manifest, indent=2), encoding='utf-8'
    )
    print('wrote:', strict_path)
    print('wrote:', ci_path)
    print('rows:', len(strict_rows), '| seeds:', len(FROZEN_SEEDS))
else:
    strict_rows = ci_rows = []
    print('No results written because required inputs are missing.')

In [ ]:
if RUN_ANALYSIS:
    preview = pd.DataFrame(strict_rows)
    print(preview.head(12).to_string(index=False))
    print('Output directory:', OUTPUT_ROOT.resolve())
else:
    print('BLOCKED: upload the exact paths listed in input_validation.json, then rerun top-to-bottom.')